# Model Development – Architecture Experiments

> Goal: benchmark multiple model families for SDRF metadata prediction from publication text using the preprocessed training set.

This notebook is an experimentation workspace rather than the final training pipeline. It focuses on comparative modeling decisions:
- a compact transformer baseline
- a long-context transformer for full-length publication text
- a Qwen-based model to test modern decoder-style representations
- a CNN text classifier as a lightweight non-transformer baseline

The current repository training code is still a placeholder, so the notebook keeps the experiments self-contained and produces artifacts that can later be folded into the package.

In [1]:
from __future__ import annotations

import math
import os
import random
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 40)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT_DIR = ROOT / 'outputs'
TRAIN_PATH = OUTPUT_DIR / 'train_preprocessed.csv'
EXPERIMENT_DIR = OUTPUT_DIR / 'models' / 'architecture_experiments'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Experiment directory: {EXPERIMENT_DIR}')

c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Experiment directory: c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\models\architecture_experiments


In [3]:
train = pd.read_csv(TRAIN_PATH, dtype=str).fillna('Not Applicable')

raw_file_col = next(
    (col for col in ['Raw Data File', 'raw_data_file', 'RawDataFile'] if col in train.columns),
    None,
)

excluded_cols = {'row_id', 'PXD', 'pub_text'}
if raw_file_col is not None:
    excluded_cols.add(raw_file_col)

label_cols = [col for col in train.columns if col not in excluded_cols]

text_prefix = 'PXD: ' + train['PXD'].astype(str)
if raw_file_col is not None:
    text_prefix = text_prefix + '\nRaw data file: ' + train[raw_file_col].astype(str)

train['text_input'] = (
    text_prefix + '\n\n' +
    train['pub_text'].astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.slice(0, 16000)
 )

label_profile = pd.DataFrame({
    'non_na_rows': (train[label_cols] != 'Not Applicable').sum(),
    'n_unique': train[label_cols].nunique(),
    'top_label_share_pct': [
        round(
            train[col].value_counts(normalize=True, dropna=False).iloc[0] * 100,
            1,
        )
        for col in label_cols
    ],
}).sort_values(['non_na_rows', 'n_unique'], ascending=[False, True])

print(f'Rows: {len(train):,}')
print(f'Candidate label columns: {len(label_cols)}')
print(f'Raw file column detected: {raw_file_col}')
display(label_profile.head(15))

Rows: 36,280
Candidate label columns: 16
Raw file column detected: None


,non_na_rows,n_unique,top_label_share_pct
Characteristics[Organism],36280,19,68.6
Comment[Instrument],36280,339,13.4
Comment[FractionIdentifier],36125,118,33.8
Characteristics[Modification],36064,3358,11.5
Characteristics[BiologicalReplicate],34437,44,81.1
Comment[FragmentMassTolerance],34275,26,20.9
Comment[PrecursorMassTolerance],34209,14,31.2
Characteristics[Disease],34062,128,15.6
Characteristics[OrganismPart],32565,135,13.5
Characteristics[Sex],31774,9,51.1


## Experiment design

> Why a single target column first?

> The repository does not yet implement the full multi-field structured extraction objective in a trainable form. To make architecture comparisons meaningful right now, we treat one SDRF column at a time as a document classification task.

This gives us a controlled way to answer questions such as:
- Does longer context help materially over a standard 512-token model?
- Does Qwen behave competitively on this scientific text?
- How much quality do we lose if we replace transformers with a much cheaper CNN?

Once a winning architecture emerges, the same backbone can be adapted into a multi-column or sequence-labeling pipeline.

In [5]:
TARGET_COL = 'Comment[Instrument]' if 'Comment[Instrument]' in label_cols else label_cols[0]
DROP_NOT_APPLICABLE = True
TOP_K_CLASSES = 12
MIN_CLASS_SUPPORT = 8
MAX_ROWS = 1500  # keep notebook experiments practical on a laptop
RANDOM_STATE = 42

experiment_df = train.loc[:, ['text_input', TARGET_COL]].copy()
experiment_df.columns = ['text_input', 'label']

if DROP_NOT_APPLICABLE:
    experiment_df = experiment_df[experiment_df['label'] != 'Not Applicable'].copy()

label_counts = experiment_df['label'].value_counts()
kept_labels = label_counts[label_counts >= MIN_CLASS_SUPPORT].head(TOP_K_CLASSES).index.tolist()
experiment_df = experiment_df[experiment_df['label'].isin(kept_labels)].copy()

if experiment_df.empty:
    raise ValueError(
        f'No rows available for {TARGET_COL} after filtering. Lower MIN_CLASS_SUPPORT or change TARGET_COL.'
    )

if len(experiment_df) > MAX_ROWS:
    cap_per_class = max(2, math.ceil(MAX_ROWS / max(len(kept_labels), 1)))
    experiment_df = (
        experiment_df
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .groupby('label', group_keys=False)
        .head(cap_per_class)
        .reset_index(drop=True)
    )

categories = pd.Categorical(experiment_df['label'])
experiment_df['label_id'] = categories.codes
id_to_label = dict(enumerate(categories.categories))
label_to_id = {label: idx for idx, label in id_to_label.items()}

train_df, val_df = train_test_split(
    experiment_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=experiment_df['label_id'],
)

print(f'Target column: {TARGET_COL}')
print(f'Training rows: {len(train_df):,} | Validation rows: {len(val_df):,}')
print(f'Classes: {len(label_to_id)}')
display(experiment_df['label'].value_counts().rename('count').to_frame())

Target column: Comment[Instrument]
Training rows: 1,200 | Validation rows: 300
Classes: 12


,count
label,
AC=MS:1002634;NT=Q Exactive Plus,125
AC=MS:1002732;NT=Orbitrap Fusion Lumos,125
NT=Q Exactive;AC=MS:1001911,125
NT=Q Exactive Plus;AC=MS:1001911,125
AC=MS:1001742;NT=LTQ Orbitrap Velos,125
NT=Q Exactive HF;AC=MS:1002523,125
AC=MS:1000639;NT=Orbitrap Fusion,125
AC=MS:1001910;NT=LTQ Orbitrap Elite,125
Q Exactive Plus,125


In [6]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SequenceClassificationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.texts = frame['text_input'].tolist()
        self.labels = frame['label_id'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class TextCNNClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_labels: int, embed_dim: int = 256, num_filters: int = 128, kernel_sizes=(3, 5, 7), dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_labels)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)
        features = [torch.amax(torch.relu(conv(x)), dim=2) for conv in self.convs]
        x = torch.cat(features, dim=1)
        x = self.dropout(x)
        return self.classifier(x)


class SimpleVocab:
    def __init__(self, min_freq: int = 2, max_vocab: int = 30000):
        self.min_freq = min_freq
        self.max_vocab = max_vocab
        self.pad_id = 0
        self.unk_id = 1
        self.token_to_id = {'<pad>': self.pad_id, '<unk>': self.unk_id}

    @staticmethod
    def tokenize(text: str) -> list[str]:
        return str(text).lower().split()

    def fit(self, texts: list[str]) -> None:
        counter = Counter()
        for text in texts:
            counter.update(self.tokenize(text))
        for token, freq in counter.most_common(self.max_vocab):
            if freq < self.min_freq or token in self.token_to_id:
                continue
            self.token_to_id[token] = len(self.token_to_id)

    def encode(self, text: str, max_length: int) -> list[int]:
        tokens = self.tokenize(text)[:max_length]
        ids = [self.token_to_id.get(token, self.unk_id) for token in tokens]
        if len(ids) < max_length:
            ids.extend([self.pad_id] * (max_length - len(ids)))
        return ids

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)


class CNNDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, vocab: SimpleVocab, max_length: int):
        self.texts = frame['text_input'].tolist()
        self.labels = frame['label_id'].tolist()
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        return {
            'input_ids': torch.tensor(self.vocab.encode(self.texts[idx], self.max_length), dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [7]:
@dataclass
class ExperimentSpec:
    name: str
    family: str
    model_name: str | None = None
    max_length: int = 512
    batch_size: int = 4
    learning_rate: float = 2e-5
    epochs: int = 1


def compute_metrics(y_true: list[int], y_pred: list[int]) -> dict:
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
    }


def evaluate_hf(model, loader, family: str) -> dict:
    model.eval()
    predictions = []
    targets = []
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            if family == 'longformer':
                batch['global_attention_mask'] = torch.zeros_like(batch['attention_mask'])
                batch['global_attention_mask'][:, 0] = 1
            outputs = model(**batch)
            losses.append(outputs.loss.item())
            predictions.extend(outputs.logits.argmax(dim=1).cpu().tolist())
            targets.extend(batch['labels'].cpu().tolist())
    metrics = compute_metrics(targets, predictions)
    metrics['loss'] = float(np.mean(losses)) if losses else np.nan
    return metrics


def train_hf_experiment(spec: ExperimentSpec, train_df: pd.DataFrame, val_df: pd.DataFrame) -> dict:
    set_seed(RANDOM_STATE)
    tokenizer = AutoTokenizer.from_pretrained(spec.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token

    train_dataset = SequenceClassificationDataset(train_df, tokenizer, spec.max_length)
    val_dataset = SequenceClassificationDataset(val_df, tokenizer, spec.max_length)

    train_loader = DataLoader(train_dataset, batch_size=spec.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=spec.batch_size, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        spec.model_name,
        num_labels=len(label_to_id),
        trust_remote_code=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=spec.learning_rate)
    history = []

    for epoch in range(spec.epochs):
        model.train()
        running_losses = []
        progress = tqdm(train_loader, desc=f'{spec.name} epoch {epoch + 1}', leave=False)
        for batch in progress:
            batch = {key: value.to(device) for key, value in batch.items()}
            if spec.family == 'longformer':
                batch['global_attention_mask'] = torch.zeros_like(batch['attention_mask'])
                batch['global_attention_mask'][:, 0] = 1
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_losses.append(loss.item())
            progress.set_postfix(loss=np.mean(running_losses))

        val_metrics = evaluate_hf(model, val_loader, spec.family)
        val_metrics['train_loss'] = float(np.mean(running_losses)) if running_losses else np.nan
        val_metrics['epoch'] = epoch + 1
        history.append(val_metrics)

    best_metrics = max(history, key=lambda row: row['macro_f1'])
    best_metrics['name'] = spec.name
    best_metrics['family'] = spec.family
    best_metrics['model_name'] = spec.model_name
    best_metrics['max_length'] = spec.max_length
    best_metrics['batch_size'] = spec.batch_size
    return best_metrics

In [8]:
def evaluate_cnn(model, loader) -> dict:
    model.eval()
    predictions = []
    targets = []
    losses = []
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids)
            loss = loss_fn(logits, labels)
            losses.append(loss.item())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    metrics = compute_metrics(targets, predictions)
    metrics['loss'] = float(np.mean(losses)) if losses else np.nan
    return metrics


def train_cnn_experiment(spec: ExperimentSpec, train_df: pd.DataFrame, val_df: pd.DataFrame) -> dict:
    set_seed(RANDOM_STATE)
    vocab = SimpleVocab(min_freq=2, max_vocab=40000)
    vocab.fit(train_df['text_input'].tolist())

    train_dataset = CNNDataset(train_df, vocab, spec.max_length)
    val_dataset = CNNDataset(val_df, vocab, spec.max_length)

    train_loader = DataLoader(train_dataset, batch_size=spec.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=spec.batch_size, shuffle=False)

    model = TextCNNClassifier(vocab_size=vocab.vocab_size, num_labels=len(label_to_id))
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=spec.learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for epoch in range(spec.epochs):
        model.train()
        running_losses = []
        progress = tqdm(train_loader, desc=f'{spec.name} epoch {epoch + 1}', leave=False)
        for batch in progress:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids)
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_losses.append(loss.item())
            progress.set_postfix(loss=np.mean(running_losses))

        val_metrics = evaluate_cnn(model, val_loader)
        val_metrics['train_loss'] = float(np.mean(running_losses)) if running_losses else np.nan
        val_metrics['epoch'] = epoch + 1
        history.append(val_metrics)

    best_metrics = max(history, key=lambda row: row['macro_f1'])
    best_metrics['name'] = spec.name
    best_metrics['family'] = spec.family
    best_metrics['model_name'] = 'TextCNN'
    best_metrics['max_length'] = spec.max_length
    best_metrics['batch_size'] = spec.batch_size
    return best_metrics

## Architecture registry

> Suggested starting point on CPU: run `textcnn` first, then `distilbert`.

> Longformer and Qwen are included because they are the architectures most likely to expose gains from full-document context and stronger representations, but they are much heavier on memory and download time.

In [9]:
experiments = [
    ExperimentSpec(
        name='textcnn',
        family='cnn',
        model_name=None,
        max_length=512,
        batch_size=32,
        learning_rate=1e-3,
        epochs=3,
    ),
    ExperimentSpec(
        name='distilbert',
        family='transformer',
        model_name='distilbert-base-uncased',
        max_length=512,
        batch_size=8,
        learning_rate=2e-5,
        epochs=1,
    ),
    ExperimentSpec(
        name='longformer',
        family='longformer',
        model_name='allenai/longformer-base-4096',
        max_length=1024,
        batch_size=1,
        learning_rate=2e-5,
        epochs=1,
    ),
    ExperimentSpec(
        name='qwen2_5_0_5b',
        family='qwen',
        model_name='Qwen/Qwen2.5-0.5B',
        max_length=1024,
        batch_size=1,
        learning_rate=1e-5,
        epochs=1,
    ),
]

experiment_table = pd.DataFrame([spec.__dict__ for spec in experiments])
display(experiment_table)

selected_experiment_names = ['textcnn']
selected_experiments = [spec for spec in experiments if spec.name in selected_experiment_names]

print('Selected experiments:')
display(pd.DataFrame([spec.__dict__ for spec in selected_experiments]))

,name,family,model_name,max_length,batch_size,learning_rate,epochs
0,textcnn,cnn,NaN,512,32,0.00100,3
1,distilbert,transformer,distilbert-base-uncased,512,8,0.00002,1
2,longformer,longformer,allenai/longformer-base-4096,1024,1,0.00002,1
3,qwen2_5_0_5b,qwen,Qwen/Qwen2.5-0.5B,1024,1,0.00001,1


Selected experiments:


,name,family,model_name,max_length,batch_size,learning_rate,epochs
0,textcnn,cnn,None,512,32,0.001,3


In [ ]:
results = []

for spec in selected_experiments:
    start_time = time.time()
    print(f'\nRunning experiment: {spec.name}')
    try:
        if spec.family == 'cnn':
            result = train_cnn_experiment(spec, train_df, val_df)
        else:
            result = train_hf_experiment(spec, train_df, val_df)
        result['status'] = 'ok'
        result['error'] = ''
    except Exception as exc:
        result = {
            'name': spec.name,
            'family': spec.family,
            'model_name': spec.model_name or 'TextCNN',
            'max_length': spec.max_length,
            'batch_size': spec.batch_size,
            'accuracy': np.nan,
            'macro_f1': np.nan,
            'weighted_f1': np.nan,
            'loss': np.nan,
            'status': 'failed',
            'error': str(exc)[:300],
        }
    result['minutes'] = round((time.time() - start_time) / 60, 2)
    results.append(result)

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values(
        by=['status', 'macro_f1'],
        ascending=[True, False],
        na_position='last',
    ).reset_index(drop=True)

results_path = EXPERIMENT_DIR / f"{TARGET_COL.replace('[', '_').replace(']', '')}_architecture_results.csv"
results_df.to_csv(results_path, index=False)
print(f'Results saved to: {results_path}')
display(results_df)


Running experiment: textcnn


textcnn epoch 3:   8%|▊         | 3/38 [00:02<00:24,  1.42it/s, loss=0.472] 

In [ ]:
successful_results = results_df[results_df['status'] == 'ok'].copy()

if successful_results.empty:
    print('No successful experiment results yet. Inspect the error column above.')
else:
    display(successful_results[['name', 'family', 'accuracy', 'macro_f1', 'weighted_f1', 'minutes']])

    ax = successful_results.sort_values('macro_f1').plot(
        x='name',
        y=['macro_f1', 'weighted_f1', 'accuracy'],
        kind='barh',
        figsize=(10, 5),
    )
    ax.set_title(f'Architecture comparison for {TARGET_COL}')
    ax.set_xlabel('Score')
    ax.set_ylabel('Experiment')
    plt.tight_layout()
    plt.show()

    best_row = successful_results.sort_values('macro_f1', ascending=False).iloc[0]
    print('Best successful run:')
    print(best_row[['name', 'family', 'model_name', 'macro_f1', 'weighted_f1', 'minutes']].to_string())

## What to do with the results

> If `textcnn` is close to transformer quality, it is a strong candidate for a fast fallback or ensemble component.

> If `longformer` materially beats `distilbert`, the task is probably context-limited rather than label-limited.

> If `qwen2_5_0_5b` wins, the next step is to move from column-wise classification to structured multi-field prediction with the same backbone.

> After you identify a winner for one target column, repeat the notebook with harder columns such as `Characteristics[Modification]` and `Characteristics[Disease]` before committing to package-level refactors.